# TusoAI scDRS+ repo runner

This notebook configures TusoAI to optimize functions in the scDRS+ example codebase.


In [ ]:
import os
from tusoai import Tusoai

# Set OPENAI_API_KEY in your environment before running this notebook.
# Set SEMANTIC_SCHOLAR_API_KEY optionally for higher-throughput literature search.
semantic_scholar_api_key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY")

ai = Tusoai.from_api_key(
    api_key=os.environ["OPENAI_API_KEY"],
    provider="openai",
    temperature=1.0,
    max_tokens=15000,
    model_settings={
        # PDF parsing / summarization model
        "pdf": {
            "model": "gpt-5.4-nano",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
        # Knowledge-tree/instruction construction model
        "construction": {
            "model": "gpt-5.4",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
        # Optimization/mutation model
        "optimization": {
            "model": "gpt-5.4-nano",
            "thinking": True,
            "thinking_tokens": 5000,
            "reasoning_mode": "medium",
        },
    },
)


In [ ]:
task_description = "associating cells to disease through gene expression"
data_available = "scRNA-seq data and a disease gene set"
cache_dir = "tusoai_scdrsplus"

# --- Build the method subtask ---
function_name = "fit_conditional_effects_with_evidence_shrinkage"

method_task, method_cost = ai.create_method_subtask(
    function_name=function_name,
    task_description=task_description,
    data_available=data_available,
    num_cat=10,
    instruction_count=30,
    num_init=10,
    paper_searches=10,
    info_per_paper=20,
    clear=False,
    cache_dir=cache_dir,
    semantic_scholar_api_key=semantic_scholar_api_key,
    hints=[
        "Keep your implementation principled, concise, and efficient.",
        "Do not introduce additional hyperparameters.",
        "This function assess the conditional association of each cell on each other.",
        "Keep the function header and output shape unchanged.",
    ],
    use_initial=True,
    source_path="conditional_analysis.py",
    repo_root="scdrsplus/scdrsplus",
)


In [ ]:
task_description = "associating cells to disease through gene expression"
data_available = "scRNA-seq data and a disease gene set"
cache_dir = "tusoai_scdrsplus"

# --- Build the method subtask ---
function_name = "fit_marginal_association_effects"

method_task2, method_cost = ai.create_method_subtask(
    function_name=function_name,
    task_description=task_description,
    data_available=data_available,
    num_cat=10,
    instruction_count=30,
    num_init=10,
    paper_searches=10,
    info_per_paper=20,
    clear=False,
    cache_dir=cache_dir,
    semantic_scholar_api_key=semantic_scholar_api_key,
    hints=[
        "Keep your implementation principled, concise, and efficient.",
        "Do not introduce additional hyperparameters.",
        "This function assess the marginal association of a cell to disease.",
        "Keep the function header and output shape unchanged.",
    ],
    use_initial=True,
    source_path="marginal_analysis.py",
    repo_root="scdrsplus/scdrsplus",
)


In [ ]:

# --- Run discovery ---
best_model, history = ai.optimize(
    method_tasks=[method_task, method_task2],
    data_tasks=[],
    reference_filename="scdrsplus/scdrsfm_runner.py",
    timeout=300,
    bug_retries=3,
    n_feedback_buffer=15,
    skip_timeout=True,
    prompt_samples=3,
    drop_island_iter=60,
    prompt_decay=2.0,
    prompt_importance=100.0,
    max_islands=3,
    output_dir=cache_dir,
    TIME_LIMIT=72 * 60,
    task_description=task_description,
    debug=True,
    min_improvement=0.005,
    n_jobs=1,
    COST_LIMIT=20,
)
